# Face Scanner — GPU Worker (Google Colab)

Run cells top-to-bottom. Re-run **Cell 4** if the worker crashes.

**Requirements:** Runtime → Change runtime type → T4 GPU

In [ ]:
!pip install -q "fastapi>=0.115.2,<1.0" "uvicorn[standard]>=0.30,<1.0" "python-multipart>=0.0.18" insightface "onnxruntime-gpu>=1.19" "numpy>=2.0" Pillow pyngrok aiofiles
print("✅ Done")

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

os.environ['WORKER_DB_PATH'] = '/content/drive/MyDrive/face_scanner/embeddings.db'
os.environ['FACE_THRESHOLD'] = '0.65'
os.environ['RAILWAY_URL'] = 'https://YOUR-RAILWAY-APP.railway.app'  # change this
os.environ['WORKER_SECRET'] = 'change-this-shared-secret'

print("✅ Config set")

In [ ]:
import subprocess, os
REPO_URL = 'https://github.com/GuyyaReiki/face-scanner.git'
WORKER_DIR = '/content/face_scanner_worker'
if os.path.exists(f'{WORKER_DIR}/.git'):
    subprocess.run(['git', '-C', WORKER_DIR, 'pull'], capture_output=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, WORKER_DIR], capture_output=True)
print(f"✅ Code at {WORKER_DIR}")

In [ ]:
import subprocess, os, time, json, urllib.request, builtins

WORKER_DIR = '/content/face_scanner_worker/colab_worker'
LOG = '/tmp/worker.log'

if hasattr(builtins, '_worker_proc'):
    old = builtins._worker_proc
    if old.poll() is None:
        old.terminate(); old.wait(timeout=5)
        print('🔄 Stopped old worker'); time.sleep(1)
subprocess.run(['fuser', '-k', '8001/tcp'], capture_output=True); time.sleep(1)

log_fh = open(LOG, 'w')
proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'worker:app', '--host', '0.0.0.0', '--port', '8001', '--log-level', 'info'],
    cwd=WORKER_DIR, env=os.environ.copy(), stdout=log_fh, stderr=log_fh
)
builtins._worker_proc = proc
print(f'🚀 Worker PID {proc.pid}')

# Wait for ready
deadline = time.time() + 120
while time.time() < deadline:
    if proc.poll() is not None:
        print(open(LOG).read()[-2000:]); break
    try:
        resp = urllib.request.urlopen('http://localhost:8001/health', timeout=3)
        print(f'✅ Worker ready — {json.loads(resp.read())}'); break
    except:
        print('.', end='', flush=True); time.sleep(3)

In [ ]:
from pyngrok import ngrok, conf
import getpass, requests, os

NGROK_TOKEN = getpass.getpass('ngrok authtoken: ')
conf.get_default().auth_token = NGROK_TOKEN

url = ngrok.connect(8001)
worker_url = str(url)
print(f'🌐 Worker URL: {worker_url}')

# Notify Railway so it knows where the worker is
RAILWAY_URL = os.environ.get('RAILWAY_URL', '')
SECRET = os.environ.get('WORKER_SECRET', '')
if RAILWAY_URL and RAILWAY_URL != 'https://YOUR-RAILWAY-APP.railway.app':
    try:
        r = requests.post(f'{RAILWAY_URL}/api/internal/register-worker',
            json={'url': worker_url, 'secret': SECRET}, timeout=10)
        print(f'✅ Registered with Railway: {r.status_code}')
    except Exception as e:
        print(f'⚠️  Could not register: {e}')
        print(f'   Set RECOGNIZER_URL={worker_url} manually in Railway env vars')
else:
    print(f'⚠️  Set RECOGNIZER_URL={worker_url} in Railway environment variables')

In [ ]:
import time, urllib.request, json
from datetime import datetime
print('🔄 Keep-alive — press ■ to stop')
while True:
    try:
        urllib.request.urlopen('http://localhost:8001/health', timeout=5)
    except Exception as e:
        print(f'[{datetime.now().strftime("%H:%M")}] ⚠️ {e}')
    time.sleep(300)